In [7]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

In [8]:
# Ollama exposes an OpenAI-compatible API — point directly at it
model = OpenAIModel(
    model_name="gemma4:26b",
    provider=OpenAIProvider(base_url="http://localhost:11434/v1"),
)

print("Model configured:", model)

Model configured: OpenAIModel()


/var/folders/6k/3yb_sy2d4plffrxc253561_40000gp/T/ipykernel_89812/896140090.py:2: DeprecationWarning: `OpenAIModel` was renamed to `OpenAIChatModel` to clearly distinguish it from `OpenAIResponsesModel` which uses OpenAI's newer Responses API. Use that unless you're using an OpenAI Chat Completions-compatible API, or require a feature that the Responses API doesn't support yet like audio.
  model = OpenAIModel(


In [9]:
from pydantic import BaseModel
class ResearchResult(BaseModel):
    summary: str
    key_points: list[str]
    confidence: float  # 0.0 – 1.0

In [16]:
agent = Agent(
    model=model,
    output_type=ResearchResult,
    system_prompt=(
        "You are a teaching assistant. "
        "Analyse the user text in German and return the structured grammar summary." \
        "The grammar summary should include the obeyed rules with the explanation of the rules and the number of times the rules are obeyed in the text." \
        "Each rule should be explained in a simple way and include an example of the error and the correction. But the example is not taken from the text." \
        "No indication where the errors in text appear should be given. " \
        "The user should correct the text by himself and learn from the grammar summary. " \
        "The user should input the corrected text to the agent" \
        "The agent should check the corrected text and return the grammar summary again. " \
        "In case the text is big (> 250 words), the agent should split the text into smaller parts and return the grammar summary for each part. " \
        "In case several errors obey the same rule, cluster them and show the obeyed rule only once in the summary. " \
        "All errors, the rules they obey need to stored in a memory of the agent" \
        "The memory of the agent is stored in a vector database and can be accessed by the agent at any time. " \
        "The agent should stop to produce the grammar summary if the user input is correct , if the user input is not in German, or if the user input is empty. " \
        "The agent should produce the grammar summary only if the user input is in German and contains grammar errors. " \
        "The agent must stop if the user input includes the word 'stop' or 'exit'. " \
        "The agent should stop to create grammar summary after 5 iterations of user input and grammar summary on the same text. " \
        "The user text is: {input}" \
        "The output should be in German." \
    ),
)

In [11]:
@agent.tool_plain
def memory_search(query: str) -> str:
    """Search the agent's memory for relevant information."""
    # In a real implementation, this would query a vector database or similar
    return f"Memory search results for query: '{query}'"

@agent.tool_plain
def store_errors_in_memory(errors: list[str]) -> str:
    """Store identified errors in the agent's memory."""
    # In a real implementation, this would store the errors in a vector database or similar
    return f"Stored errors in memory: {', '.join(errors)}"

In [18]:
async def run_agent(query: str) -> ResearchResult:
    result = await agent.run(query)
    return result.output

# Jupyter has a running event loop, so use await directly
output = await run_agent(
   "heute bin ich zum See gefahren um dort zu schwmimmen. ich habe eine halbe Stunde in Wasser gewesen. "
)

print(output.model_dump_json(indent=2))

{
  "summary": "**Regel: Großschreibung am Satzanfang**\n* Erklärung: Jeder neue Satz muss im Deutschen mit einem Großbuchstaben beginnen.\n* Anzahl der Verstöße: 2\n* Beispiel Fehler: „der tag war schön.“ $\\rightarrow$ Korrektur: „Der Tag war schön.“\n\n**Regel: Rechtschreibung**\n* Erklärung: Wörter müssen nach den Regeln der deutschen Orthografie korrekt geschrieben werden.\n* Anzahl der Verstöße: 1\n* Beispiel Fehler: „Ich liebe die Sonneee.“ $\\rightarrow$ Korrektur: „Ich liebe die Sonne.“\n\n**Regel: Kommasetzung bei Infinitivgruppen (um...zu)**\n* Erklärung: Vor einer Infinitivgruppe, die mit „um“ eingeleitet wird, muss ein Komma gesetzt werden.\n* Anzahl der Verstöße: 1\n* Beispiel Fehler: „Ich gehe in den Park um zu joggen.“ $\\rightarrow$ Korrektur: „Ich gehe in den Park, um zu joggen.“\n\n**Regel: Hilfsverb „sein“ im Perfekt**\n* Erklärung: Verben, die eine Ortsveränderung oder eine Zustandsänderung beschreiben, bilden das Perfekt mit dem Hilfsverb „sein“.\n* Anzahl der Ver

In [19]:
# Multi-turn conversation
from pydantic_ai.messages import ModelMessagesTypeAdapter

history = []

async def chat(user_input: str) -> str:
    global history
    result = await agent.run(user_input, message_history=history)
    history += result.new_messages()  # append this turn to history
    return result.output

# Turn 1
r1 = await chat("What is Mixture-of-Experts?")
print("Turn 1:", r1)

UnexpectedModelBehavior: Exceeded maximum retries (1) for output validation